# 🎙️ SautiCivic Bridge: Whisper large-v3 Multi-Speaker Validation (Tier A-MSV)
### Sahara CodeSwitch Africa Challenge 2026 — Legal & Public Services Track

This notebook executes empirical benchmarking of **OpenAI Whisper large-v3** on the **30 Tier A Multi-Speaker Validation (MSV)** audio recordings across three previously unseen Nigerian speakers:
- **`SPK-03`**: **Chinenye** (Female, Igbo-substrate Nigerian Pidgin) — `spk3_f_001` to `spk3_f_010`
- **`SPK-04`**: **Daniel** (Male, standard Nigerian Pidgin) — `spk4_m_001` to `spk4_m_010`
- **`SPK-05`**: **Mukhtar** (Male, Northern-substrate Nigerian Pidgin) — `spk5_m_001` to `spk5_m_010`

---

### 📋 Pipeline Workflow:
1. **Mount Google Drive** to access the raw `.ogg` recordings.
2. **Auto-Locate or Specify** the three speaker folders (`Chineye tier a audio`, `Daniel tier a audio`, `Mukthar tier a audio`).
3. **Convert `.ogg` audio to canonical 16kHz mono WAV** with ffmpeg (exact benchmark specification).
4. **Load OpenAI Whisper `large-v3`** onto GPU (T4 / A100).
5. **Transcribe all 30 clips** and save schema-compliant JSON transcripts matching the SautiCivic repository format.
6. **Evaluate Aspectual Polarity Inversion** (*don* $	o$ *don't*) & **Word Error Rate (WER)**.
7. **Display Cross-Model Comparison** against Sahara v2.5, Gemini 3.5 Transcribe, and Deepgram Nova-3.
8. **Package & Download** all 30 JSON files as a `.zip` archive for immediate commitment to `bench/results/transcripts/tier_a_multispeaker_validation/whisper/`.

In [ ]:
# ==============================================================================
# 1. MOUNT GOOGLE DRIVE & INSTALL DEPENDENCIES
# ==============================================================================
from google.colab import drive
import os, sys, shutil, json, time, re, hashlib, subprocess
from pathlib import Path
from datetime import datetime, timezone
from collections import Counter

# Mount Google Drive
print("Mounting Google Drive...")
drive.mount('/content/drive')

# Install OpenAI Whisper, SoundFile, jiwer, and ffmpeg
print("\nInstalling OpenAI Whisper, SoundFile, jiwer, and audio dependencies...")
!pip install -q openai-whisper soundfile jiwer ffmpeg-python torch torchvision torchaudio
!apt-get install -y ffmpeg -qq

print("\n Dependencies installed successfully!")

In [ ]:
# ==============================================================================
# 2. VERIFY GPU ACCELERATION
# ==============================================================================
import torch

cuda_available = torch.cuda.is_available()
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available:  {cuda_available}")

if cuda_available:
    device = "cuda"
    device_name = torch.cuda.get_device_name(0)
    print(f"Active GPU:      {device_name}")
    print(f"VRAM Allocated:  {torch.cuda.memory_allocated(0)/(1024**2):.1f} MB / {torch.cuda.get_device_properties(0).total_memory/(1024**2):.1f} MB")
else:
    device = "cpu"
    print("\n⚠️ WARNING: GPU is NOT detected! Whisper large-v3 inference on CPU will be slow.")
    print("👉 Enable GPU acceleration: Go to Runtime > Change runtime type > Select 'T4 GPU' > Save.")

In [ ]:
# ==============================================================================
# 3. LOCATE SPEAKER FOLDERS ON GOOGLE DRIVE (.ogg format)
# ==============================================================================
# The files are organized in 3 folders on Drive:
#   1. "Chineye tier a audio"
#   2. "Daniel tier a audio"
#   3. "Mukthar tier a audio"

# If your folders are in a specific subdirectory, you can specify it here:
# Example: CUSTOM_SEARCH_PATH = "/content/drive/MyDrive/Sauticivic"
# Set to None to automatically search under /content/drive/MyDrive:
CUSTOM_SEARCH_PATH = None

TARGET_FOLDER_NAMES = [
    "Chineye tier a audio",
    "Daniel tier a audio",
    "Mukthar tier a audio"
]

def find_speaker_folders(base_search_dir="/content/drive/MyDrive"):
    found = {}
    base_path = Path(base_search_dir)
    print(f"Searching for speaker folders inside: {base_path} (this may take a few seconds)...")
    
    # Check common known locations first for speed
    common_candidates = [
        base_path,
        base_path / "Sauticivic",
        base_path / "Sauticivic" / "bench" / "corpus" / "tier_a_recorded",
        base_path / "sauticivic-bridge" / "bench" / "corpus" / "tier_a_recorded",
    ]
    for cand in common_candidates:
        if cand.is_dir():
            for name in TARGET_FOLDER_NAMES:
                p = cand / name
                if p.is_dir() and name not in found:
                    found[name] = p
                    
    # If not all found, do recursive search
    if len(found) < len(TARGET_FOLDER_NAMES):
        for root, dirs, _ in os.walk(base_path):
            for d in dirs:
                if d in TARGET_FOLDER_NAMES and d not in found:
                    found[d] = Path(root) / d
            if len(found) == len(TARGET_FOLDER_NAMES):
                break
                
    return found

search_root = CUSTOM_SEARCH_PATH if CUSTOM_SEARCH_PATH else "/content/drive/MyDrive"
located_paths = find_speaker_folders(search_root)

print("\nSearch Results:")
folder_info = {}
for name in TARGET_FOLDER_NAMES:
    path = located_paths.get(name)
    if path and path.is_dir():
        ogg_files = list(path.glob("*.ogg")) + list(path.glob("*.OGG"))
        wav_files = list(path.glob("*.wav")) + list(path.glob("*.WAV"))
        folder_info[name] = {"path": path, "ogg": len(ogg_files), "wav": len(wav_files)}
        print(f"  ✓ {name}: {path} ({len(ogg_files)} .ogg, {len(wav_files)} .wav)")
    else:
        print(f"  ❌ {name}: NOT FOUND. Please check CUSTOM_SEARCH_PATH.")

missing = [n for n in TARGET_FOLDER_NAMES if n not in folder_info]
if missing:
    print(f"\n⚠️ Missing folders: {missing}")
    print("Please set CUSTOM_SEARCH_PATH directly to the directory containing these folders.")
else:
    print(f"\n🎉 All 3 speaker directories successfully found!")

In [ ]:
# ==============================================================================
# 4. CANONICAL MSV PROMPT MAPPING & CONVERSION (OGG -> 16kHz Mono WAV)
# ==============================================================================
SPEAKERS = {
    "Chineye tier a audio": {"speaker_id": "SPK-03", "name": "Chinenye", "gender": "female", "prefix": "spk3_f"},
    "Daniel tier a audio":  {"speaker_id": "SPK-04", "name": "Daniel",   "gender": "male",   "prefix": "spk4_m"},
    "Mukthar tier a audio": {"speaker_id": "SPK-05", "name": "Mukhtar",  "gender": "male",   "prefix": "spk5_m"},
}

PROMPTS = {
    1:  ("synth_001", "There is a big pothole for Allen Avenue junction, e don spoil plenty tyre.", 1, "affirmative", "infrastructure", "routed", "polarity_stress"),
    2:  ("synth_002", "Water don burst for our street since morning, everywhere don flood.", 2, "affirmative", "infrastructure", "routed", "polarity_stress"),
    3:  ("synth_003", "The streetlight for Ojota junction no dey work, e don dark well well.", 1, "affirmative", "infrastructure", "routed", "polarity_stress"),
    4:  ("synth_006", "My boss don sack me without paying my three months salary and overtime.", 1, "affirmative", "legal", "routed", "polarity_stress"),
    5:  ("synth_024", "My husband don chase me and the children comot for house, refuse to give us money for feeding.", 1, "affirmative", "legal", "routed", "polarity_stress"),
    6:  ("synth_005", "My landlord wan evict me and refuse to return my rent deposit.", 0, None, "legal", "routed", "control"),
    7:  ("synth_007", "Police officer arrested my brother for nothing, dem detain am for three days without charge.", 0, None, "legal", "routed", "control"),
    8:  ("synth_008", "My landlord refused to fix the burst pipe in the flat.", 0, None, "ambiguous", "needs_clarification", "control"),
    9:  ("synth_027", "The community leader don collect money from everybody for borehole wey never complete since two years.", 1, "affirmative", "ambiguous", "needs_clarification", "polarity_stress"),
    10: ("synth_029", "I wan report wetin happen yesterday but I no sure who go handle am.", 0, None, "ambiguous", "needs_clarification", "control"),
}

WORK_AUDIO_DIR = Path("/content/msv_audio_16k")
WORK_AUDIO_DIR.mkdir(parents=True, exist_ok=True)

def sha256_file(p: Path) -> str:
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

prepared_clips = []
conversion_errors = []

print(f"Converting and normalizing audio to 16kHz mono WAV in {WORK_AUDIO_DIR}...\n")

for folder_name, spk_meta in SPEAKERS.items():
    if folder_name not in folder_info:
        continue
    src_dir = folder_info[folder_name]["path"]
    
    for prompt_num, (source_id, canon_text, don_count, pol, dom, outc, grp) in PROMPTS.items():
        clip_id = f"{spk_meta['prefix']}_{prompt_num:03d}"
        target_wav = WORK_AUDIO_DIR / f"{clip_id}.wav"
        
        # Locate source audio (.ogg or .wav)
        source_candidates = [
            src_dir / f"{source_id}.ogg",
            src_dir / f"{source_id}.OGG",
            src_dir / f"{source_id}.wav",
            src_dir / f"{source_id}.WAV"
        ]
        source_file = next((c for c in source_candidates if c.exists() and c.is_file()), None)
        
        # Fallback: case-insensitive search
        if not source_file:
            for c in src_dir.glob("*"):
                if c.stem.lower() == source_id.lower() and c.suffix.lower() in [".ogg", ".wav"]:
                    source_file = c
                    break
                    
        if not source_file:
            err = f"Missing {source_id}.ogg in {src_dir}"
            print(f"  ❌ {clip_id}: {err}")
            conversion_errors.append(err)
            continue
            
        # Standardize to 16kHz, 16-bit mono PCM WAV
        if not target_wav.exists() or target_wav.stat().st_size == 0:
            cmd = [
                "ffmpeg", "-y", "-loglevel", "error",
                "-i", str(source_file),
                "-ar", "16000",
                "-ac", "1",
                "-c:a", "pcm_s16le",
                str(target_wav)
            ]
            subprocess.run(cmd, check=True)
            
        h = sha256_file(target_wav)
        prepared_clips.append({
            "clip_id": clip_id,
            "speaker_id": spk_meta["speaker_id"],
            "speaker_name": spk_meta["name"],
            "speaker_gender": spk_meta["gender"],
            "prompt_number": prompt_num,
            "source_id": source_id,
            "canonical_transcript": canon_text,
            "spoken_reference_transcript": canon_text,
            "don_token_count": don_count,
            "evaluation_group": grp,
            "expected_polarity": pol,
            "expected_domain": dom,
            "wav_path": target_wav,
            "audio_hash": h,
        })
        print(f"  ✓ {clip_id} ({spk_meta['name']}): {source_file.name} -> {target_wav.name} (SHA256: {h[:12]}...)")

print(f"\nSuccessfully prepared {len(prepared_clips)}/30 audio clips.")
if conversion_errors:
    print(f"⚠️ {len(conversion_errors)} conversion errors encountered.")

In [ ]:
# ==============================================================================
# 5. RUN WHISPER LARGE-V3 INFERENCE
# ==============================================================================
import whisper

print(f"Loading Whisper 'large-v3' model on device '{device}'...")
t_load = time.time()
model = whisper.load_model("large-v3", device=device)
print(f"✓ Whisper large-v3 loaded in {time.time() - t_load:.2f}s\n")

TRANSCRIPTS_OUT_DIR = Path("/content/whisper_msv_transcripts")
TRANSCRIPTS_OUT_DIR.mkdir(parents=True, exist_ok=True)

def normalize_text(text: str) -> str:
    return re.sub(r"[^a-z0-9' ]+", " ", text.lower()).replace("  ", " ").strip()

results = []
print(f"Transcribing {len(prepared_clips)} Tier A-MSV clips with Whisper large-v3...\n")

for i, clip in enumerate(prepared_clips, start=1):
    clip_id = clip["clip_id"]
    wav_path = clip["wav_path"]
    out_json = TRANSCRIPTS_OUT_DIR / f"{clip_id}.json"
    
    t0 = time.time()
    try:
        # Run Whisper inference (temperature 0 for deterministic greedy decoding)
        asr_res = model.transcribe(
            str(wav_path),
            task="transcribe",
            language=None, # auto-detect code-switching/language
            temperature=0.0
        )
        latency = round(time.time() - t0, 3)
        raw_text = asr_res["text"].strip()
        norm_text = normalize_text(raw_text)
        detected_lang = asr_res.get("language", "en")
        
        segments = [
            {
                "start": round(s["start"], 2),
                "end": round(s["end"], 2),
                "text": s["text"].strip(),
                "avg_logprob": round(s.get("avg_logprob", 0.0), 4),
                "no_speech_prob": round(s.get("no_speech_prob", 0.0), 4),
            }
            for s in asr_res.get("segments", [])
        ]
        
        record = {
            "clip_id": clip_id,
            "speaker_id": clip["speaker_id"],
            "model": "whisper",
            "model_identifier": "large-v3",
            "audio_hash": clip["audio_hash"],
            "raw_response": None,
            "raw_transcript": raw_text,
            "transcript": raw_text,
            "normalized_transcript": norm_text,
            "request_status": "success",
            "error": None,
            "inference_latency_s": latency,
            "language": detected_lang,
            "segments": segments,
            "timestamp": datetime.now(timezone.utc).isoformat(),
            "cached": False,
            "fresh": True
        }
        print(f"[{i:02d}/30] {clip_id} ({clip['speaker_name']}) [{latency}s | lang={detected_lang}]:")
        print(f"       \"{raw_text}\"\n")
        
    except Exception as exc:
        latency = round(time.time() - t0, 3)
        record = {
            "clip_id": clip_id,
            "speaker_id": clip["speaker_id"],
            "model": "whisper",
            "model_identifier": "large-v3",
            "audio_hash": clip["audio_hash"],
            "raw_response": None,
            "raw_transcript": "",
            "transcript": "",
            "normalized_transcript": "",
            "request_status": "failed",
            "error": str(exc),
            "inference_latency_s": latency,
            "language": None,
            "segments": [],
            "timestamp": datetime.now(timezone.utc).isoformat(),
            "cached": False,
            "fresh": True
        }
        print(f"[{i:02d}/30] {clip_id} ({clip['speaker_name']}): ❌ ERROR: {exc}\n")
        
    results.append(record)
    with open(out_json, "w", encoding="utf-8") as f:
        json.dump(record, f, indent=2, ensure_ascii=False)

print(f"\n Successfully transcribed and stored {len(results)} JSON files in {TRANSCRIPTS_OUT_DIR}!")

In [ ]:
# ==============================================================================
# 6. ASPECTUAL POLARITY ADJUDICATION (don -> don't) & WER EVALUATION
# ==============================================================================
# Exact clause-aware adjudication regexes from SautiCivic benchmark
NEGATIVE = re.compile(r"\b(?:don't|dont|doesn't|doesnt|didn't|didnt|hasn't|hasnt|haven't|havent|not|no)\b", re.I)
AFFIRMATIVE = re.compile(r"\b(?:don|done|has|have|already|is|are)\b|\b\w+(?:ed|en)\b", re.I)

def adjudicate_target(ref_phrase: str, hyp_raw: str, prompt_number: int | None = None, target_index: int = 1) -> dict:
    hyp = normalize_text(hyp_raw or "")
    ref = normalize_text(ref_phrase)
    if not hyp:
        return {"outcome": "deleted", "evidence": "", "reason": "empty transcript"}
        
    is_p2_t2 = (prompt_number == 2 and target_index == 2) or ("flood" in ref and "burst" not in ref)
    is_p2_t1 = (prompt_number == 2 and target_index == 1) or ("burst" in ref and "flood" not in ref) or ("water" in ref and "flood" not in ref)
    is_p1 = (prompt_number == 1) or ("spoil" in ref)
    is_p3 = (prompt_number == 3) or ("dark" in ref)
    is_p4 = (prompt_number == 4) or ("sack" in ref)
    is_p5 = (prompt_number == 5) or ("chase" in ref)
    is_p9 = (prompt_number == 9) or ("collect" in ref)

    if is_p2_t2:
        if re.search(r"\b(?:everywhere\s+)?(?:don't|dont|not|no)\s+(?:flood|float)\b|\bi don't float\b", hyp):
            return {"outcome": "inverted", "evidence": hyp[:120], "reason": "everywhere don flood inverted to negative"}
        if re.search(r"\b(?:everywhere\s+)?(?:don|done|has|have)\s+(?:flood|flooded)?\b|\beverywhere don\b", hyp):
            return {"outcome": "preserved", "evidence": hyp[:120], "reason": "everywhere don flood preserved"}
        return {"outcome": "ambiguous", "evidence": hyp[:120], "reason": "flood clause unaligned"}
        
    elif is_p2_t1:
        if re.search(r"\b(?:water\s+)?(?:don't|dont|not|no)\s+(?:burst|pour|pass)\b|\bi don't pass\b|\bwhat i don't post\b", hyp):
            return {"outcome": "inverted", "evidence": hyp[:120], "reason": "water don burst inverted to negative"}
        if re.search(r"\b(?:water\s+)?(?:don|done|has|have)\s+(?:burst|pour)\b|\bwater don burst\b|\bwater has burst\b", hyp):
            return {"outcome": "preserved", "evidence": hyp[:120], "reason": "water don burst preserved"}
        return {"outcome": "ambiguous", "evidence": hyp[:120], "reason": "burst clause unaligned"}
        
    elif is_p1:
        if re.search(r"\b(?:don't|dont|not|no|doesn't|doesnt|didn't|didnt)\s+(?:spoil|damage)\b|\b(?:has\s+not\s+damaged)\b", hyp) or "don't spoil" in hyp:
            return {"outcome": "inverted", "evidence": hyp[:120], "reason": "don spoil inverted to negative"}
        if re.search(r"\b(?:don|done|does|has|have|already)\s+(?:spoil|damage|damaged)\b|\b(?:e\s+don\s+spoil|don\s+spoil|has\s+damaged)\b", hyp):
            return {"outcome": "preserved", "evidence": hyp[:120], "reason": "affirmative spoil preserved"}
        if "there is a pothole" in hyp and not any(w in hyp for w in ["spoil", "tire", "tyre", "damage"]):
            return {"outcome": "deleted", "evidence": hyp[:120], "reason": "target phrase deleted"}
        return {"outcome": "ambiguous", "evidence": hyp[:120], "reason": "unaligned/ambiguous"}
        
    elif is_p3:
        if re.search(r"\b(?:you\s+)?(?:don't|dont|not|no)\s+(?:dark|duck)\b|\byou don't duck\b", hyp):
            return {"outcome": "inverted", "evidence": hyp[:120], "reason": "don dark inverted to negative"}
        if re.search(r"\b(?:e\s+)?(?:don|done|has|have)\s+dark\b", hyp):
            return {"outcome": "preserved", "evidence": hyp[:120], "reason": "don dark preserved"}
        return {"outcome": "ambiguous", "evidence": hyp[:120], "reason": "dark clause unaligned"}
        
    elif is_p4:
        if re.search(r"\b(?:boss\s+)?(?:don't|dont|not|no|doesn't|doesnt)\s+(?:sack|suck)\b", hyp):
            return {"outcome": "inverted", "evidence": hyp[:120], "reason": "don sack inverted to negative"}
        if re.search(r"\b(?:boss\s+)?(?:don|done|has|have)\s+sack\b", hyp):
            return {"outcome": "preserved", "evidence": hyp[:120], "reason": "don sack preserved"}
        return {"outcome": "ambiguous", "evidence": hyp[:120], "reason": "sack clause unaligned"}
        
    elif is_p5:
        if re.search(r"\b(?:husband\s+)?(?:don't|dont|not|no|doesn't|doesnt)\s+(?:chase|cheat)\b", hyp):
            return {"outcome": "inverted", "evidence": hyp[:120], "reason": "don chase inverted to negative"}
        if re.search(r"\b(?:husband\s+)?(?:don|done|has|have)\s+chase\b", hyp):
            return {"outcome": "preserved", "evidence": hyp[:120], "reason": "don chase preserved"}
        return {"outcome": "ambiguous", "evidence": hyp[:120], "reason": "chase clause unaligned"}
        
    elif is_p9:
        if re.search(r"\b(?:they\s+|leader\s+)?(?:don't|dont|did\s+not|not|no)\s+collect\b", hyp):
            return {"outcome": "inverted", "evidence": hyp[:120], "reason": "don collect inverted to negative"}
        if re.search(r"\b(?:leader\s+)?(?:don|done|has|have)\s+collect\b", hyp):
            return {"outcome": "preserved", "evidence": hyp[:120], "reason": "don collect preserved"}
        return {"outcome": "ambiguous", "evidence": hyp[:120], "reason": "collect clause unaligned"}

    if NEGATIVE.search(hyp):
        return {"outcome": "inverted", "evidence": hyp[:120], "reason": "negative marker in hypothesis"}
    elif re.search(r"\bdon\b|\b(?:has|have|already)\b", hyp):
        return {"outcome": "preserved", "evidence": hyp[:120], "reason": "affirmative marker present"}
    return {"outcome": "ambiguous", "evidence": hyp[:120], "reason": "polarity unestablished"}

# Adjudicate all targets
res_by_id = {r["clip_id"]: r for r in results}
adjudication_rows = []
for clip in prepared_clips:
    if clip["evaluation_group"] != "polarity_stress":
        continue
    c_id = clip["clip_id"]
    hyp_text = res_by_id.get(c_id, {}).get("raw_transcript", "")
    for idx in range(clip["don_token_count"]):
        adj = adjudicate_target(clip["canonical_transcript"], hyp_text, prompt_number=clip["prompt_number"], target_index=idx+1)
        adjudication_rows.append({
            "clip_id": c_id,
            "speaker_id": clip["speaker_id"],
            "speaker_name": clip["speaker_name"],
            "prompt_number": clip["prompt_number"],
            "target_index": idx + 1,
            "hypothesis": hyp_text,
            "outcome": adj["outcome"],
            "reason": adj["reason"]
        })

counts = Counter(r["outcome"] for r in adjudication_rows)
total_targets = 21
inverted_clips = {r["clip_id"] for r in adjudication_rows if r["outcome"] == "inverted"}
total_target_clips = 18

print("=" * 68)
print("  WHISPER LARGE-V3 ASPECTUAL POLARITY ADJUDICATION RESULTS")
print("=" * 68)
print(f"Target 'don' Tokens Evaluated: {total_targets}")
print(f"  ✓ Preserved:            {counts['preserved']} ({counts['preserved']/total_targets*100:.1f}%)")
print(f"  ❌ Inverted (Harmful):   {counts['inverted']} ({counts['inverted']/total_targets*100:.1f}%)")
print(f"  ⚪ Deleted:              {counts['deleted']} ({counts['deleted']/total_targets*100:.1f}%)")
print(f"  ❓ Ambiguous:            {counts['ambiguous']} ({counts['ambiguous']/total_targets*100:.1f}%)")
print("-" * 68)
print(f"Utterances Affected by Inversion: {len(inverted_clips)}/{total_target_clips} ({len(inverted_clips)/total_target_clips*100:.1f}%)\n")

# Per-Speaker breakdown
print("Per-Speaker Inversion Breakdown:")
for spk_id, spk_name in [("SPK-03", "Chinenye"), ("SPK-04", "Daniel"), ("SPK-05", "Mukhtar")]:
    spk_rows = [r for r in adjudication_rows if r["speaker_id"] == spk_id]
    spk_inv = sum(r["outcome"] == "inverted" for r in spk_rows)
    print(f"  • {spk_id} ({spk_name}): {spk_inv}/{len(spk_rows)} inverted ({spk_inv/len(spk_rows)*100:.1f}%)")

# Calculate Normalized WER
from jiwer import wer

def compute_wer(grp):
    selected = [c for c in prepared_clips if grp == "all" or c["evaluation_group"] == grp]
    refs = [normalize_text(c["spoken_reference_transcript"]) for c in selected]
    hyps = [normalize_text(res_by_id[c["clip_id"]]["raw_transcript"]) for c in selected]
    return wer(refs, hyps) * 100

overall_wer = compute_wer("all")
pol_wer = compute_wer("polarity_stress")
ctrl_wer = compute_wer("control")

print("\n" + "=" * 68)
print("  WORD ERROR RATE (WER) SUMMARY")
print("=" * 68)
print(f"Overall MSV WER (30 clips):       {overall_wer:.2f}%")
print(f"Polarity Stress WER (18 clips):  {pol_wer:.2f}%")
print(f"Control Group WER (12 clips):    {ctrl_wer:.2f}%")

print("\n" + "=" * 68)
print("  MULTI-SPEAKER BENCHMARK COMPARISON MATRIX (TABLE 3)")
print("=" * 68)
print(f"{'ASR Engine':<23} | {'MSV WER':<9} | {'Token Inversion':<16} | {'Utterances Affected'}")
print("-" * 68)
print(f"{'Intron Sahara v2.5':<23} | {'16.3%':<9} | {'0/21 (0.0%)':<16} | {'0/18 (0.0%)'}")
print(f"{'Gemini 3.5 Transcribe':<23} | {'28.7%':<9} | {'10/21 (47.6%)':<16} | {'9/18 (50.0%)'}")
print(f"{'Deepgram Nova-3':<23} | {'68.9%':<9} | {'10/21 (47.6%)':<16} | {'9/17 (52.9%)'}")
w_tok = f"{counts['inverted']}/21 ({counts['inverted']/21*100:.1f}%)"
w_utt = f"{len(inverted_clips)}/18 ({len(inverted_clips)/18*100:.1f}%)"
print(f"{'Whisper large-v3':<23} | {f'{overall_wer:.1f}%':<9} | {w_tok:<16} | {w_utt}")
print("=" * 68)

In [ ]:
# ==============================================================================
# 7. PACKAGE TRANSCRIPTS INTO ZIP & DOWNLOAD
# ==============================================================================
import zipfile
from google.colab import files

zip_output = Path("/content/whisper_msv_transcripts.zip")
with zipfile.ZipFile(zip_output, "w", zipfile.ZIP_DEFLATED) as zf:
    for jf in sorted(TRANSCRIPTS_OUT_DIR.glob("*.json")):
        zf.write(jf, arcname=jf.name)

print(f"✓ Packaged {len(list(TRANSCRIPTS_OUT_DIR.glob('*.json')))} JSON transcripts into {zip_output} ({zip_output.stat().st_size} bytes)")

# Optional copy directly back to Drive if destination provided
DRIVE_BACKUP_DIR = Path("/content/drive/MyDrive/sauticivic_whisper_msv_transcripts")
try:
    DRIVE_BACKUP_DIR.mkdir(parents=True, exist_ok=True)
    shutil.copy(zip_output, DRIVE_BACKUP_DIR / zip_output.name)
    print(f"✓ Also saved backup copy on Google Drive at: {DRIVE_BACKUP_DIR / zip_output.name}")
except Exception as e:
    print(f"(Drive backup skipped: {e})")

print("\nStarting browser download of whisper_msv_transcripts.zip...")
files.download(str(zip_output))